# 🚀 Loan Default Predictor: Master Pipeline

This notebook consolidates the entire project workflow into a single execution path. 
It covers:
1. **Data Loading & Cleaning**
2. **Exploratory Data Analysis (EDA)**
3. **Preprocessing & Pipeline Construction**
4. **Model Training (Ensembles: Random Forest & XGBoost)**
5. **Evaluation & Explainability (SHAP)**
6. **Model Serialization**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (f1_score, classification_report, ConfusionMatrixDisplay, 
                             accuracy_score, precision_score, recall_score)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap

# Import project modules
from src.preprocess import build_preprocessor
from src.config import (
    RAW_DATA_PATH, PROCESSED_DATA_DIR, MODEL_PATH, RESULTS_PATH,
    NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET, RANDOM_STATE
)

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

## 1. Data Loading & Cleaning
We load the raw Kaggle dataset and perform basic cleaning (dropping IDs, handling column names).

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
print(f"Initial shape: {df.shape}")

# Basic cleaning
if 'LoanID' in df.columns:
    df = df.drop(columns=['LoanID'])

print(f"Shape after dropping ID: {df.shape}")
print(f"Class Imbalance:\n{df[TARGET].value_counts(normalize=True) * 100}")
df.head()

## 2. Train-Test Split
We use stratified splitting to ensure the class imbalance (88/12) is preserved in both sets.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

## 3. Pipeline & Training (XGBoost)
We use an end-to-end Pipeline that handles scaling, encoding, and the classifier.

In [ ]:
# Build the preprocessor module
preprocessor = build_preprocessor()

# Define the XGBoost Pipeline
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        random_state=RANDOM_STATE,
        scale_pos_weight=7.3, # Calculated from imbalance ratio
        eval_metric='logloss'
    ))
])

print("Training XGBoost model...")
xgb_pipeline.fit(X_train, y_train)
print("Done.")

## 4. Evaluation
Focusing on F1-score for the positive class (defaulters).

In [ ]:
y_pred = xgb_pipeline.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues', ax=ax)
plt.title("XGBoost Confusion Matrix")
plt.show()

## 5. Explainability (SHAP)
Understanding which features contribute most to the risk prediction.

In [ ]:
# Transform data for SHAP
X_test_transformed = xgb_pipeline.named_steps['preprocessor'].transform(X_test[:100])
classifier = xgb_pipeline.named_steps['classifier']

# Get feature names
cat_encoder = xgb_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
cat_features = list(cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES))
feature_names = NUMERICAL_FEATURES + cat_features

explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_test_transformed)

shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names)

## 6. Saving the Model
Exporting the entire pipeline as a pickle file for use in the Streamlit app.

In [ ]:
os.makedirs('models', exist_ok=True)
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(xgb_pipeline, f)

print(f"Model successfully saved to: {MODEL_PATH}")